## 🎯 Practice Exercises
## Exercise 1: Build Your First Stateful Agent

**Difficulty:** Beginner
**Estimated Time:** 30-45 minutes

### Task
Build a simple customer support chatbot that remembers conversation context.

### Requirements
1. Create a StateGraph with MessagesState
2. Add a system prompt that makes the agent act as a helpful customer support rep
3. Use MemorySaver checkpointer for memory
4. Test with a multi-turn conversation where context matters

### Example Conversation
```
User: "I bought a laptop last week"
Agent: "I'd be happy to help with your laptop! What seems to be the issue?"
User: "It won't turn on"
Agent: "I understand your laptop won't turn on. Have you tried..."
```

## Step 1: Setup

In [ ]:
# # Install required packages (run once)
# %pip install -q langgraph langchain langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Import Libraries

In [4]:
from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from IPython.display import Image, display
import os
print('All libraries imported succcessfully!')

All libraries imported succcessfully!


## Step 3: Set up OpenAI API Key

In [5]:
# Load environment variables

load_dotenv()
api_key = os.getenv('api_key')

if not api_key:
    raise ValueError('API KEY NOT FOUND!')

print('API KEY loaded successfully')

API KEY loaded successfully


## Step 4: Initialize the LLM

In [6]:
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature=0.7,
    api_key=api_key
)

print('LLM Initialized: ', llm.model_name)

LLM Initialized:  gpt-4o-mini


## Step 4: Create the Node

In [23]:
# System prompt that defines the customer support behavior
system_message = SystemMessage(
    content= 'You are a friendly, calm and professional customer support rep that helps users navigate the use of a product, and fix issues they might be having. Do not hallucinate information.'
)

def assistant(state: MessagesState) -> dict:
    """
    The assistant node processes the user input and generates appropriate responses"""

    # Combinne system prompt with conversation history
    messages = [system_message] + state["messages"]

    # Get responsse from LLM
    response = llm.invoke(messages)

    # Return as state update
    return {'messages': [AIMessage(content = response.content)]}

print('Assitant Node Successfully Defined!')

Assitant Node Successfully Defined!


# Step 5: Building the Graph

In [24]:
# Create a StateGraph with MessagesState
builder = StateGraph(MessagesState)

# Add the assistant node
builder.add_node('assistant', assistant)

# Define the flow: START => assistant => END
builder.add_edge(START, "assistant")
builder.add_edge('assistant', END)

print('Graph Structure Defined Successfully')




Graph Structure Defined Successfully


## Step 6: Create Checkpointers (Memory)

In [25]:
# Create the memory checkpointer
memory = MemorySaver()

# Compile the graph with the memory
agent = builder.compile(checkpointer=memory)

print('Agent compiled with memory successfully')

Agent compiled with memory successfully


#### **Visualize the Graph**

In [26]:
# Visualize the graph structure
try: 
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f'Could not display the graph: {e}')
    print('Graph Structure: START -> assistant -> END')


Could not display the graph: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`
Graph Structure: START -> assistant -> END


## Step 7: Create Session ID

In [27]:
session_id = "chat-session-0001"

print('Session ID successfully created:', session_id)

Session ID successfully created: chat-session-0001


## Step 8: Create Helper Function for Conversations

In [32]:
def run_conversation(user_input: str, thread_id: str = session_id):
    """Send a message to the agent and get response.
    WARNING: Provide unique thread per id_user
    """

    # Invoke the agent
    result = agent.invoke(
        {'messages': [HumanMessage(content=user_input)]},
        config = {'configurable': {'thread_id': thread_id}}
    )

    # Print the conversation
    for message in result['messages']:
        if isinstance(message, HumanMessage):
            print(f'\n User: {message.content}')
        elif isinstance(message, AIMessage):
            print(f'Agent: {message.content}')

    print("\n" + "#"*70)

print('Conversation Function Successfully Created')

Conversation Function Successfully Created


#### Test running with single test

In [44]:
run_conversation('Hi. What is your name?')


 User: Hi. How are you doing today?

 User: Hi. How are you doing today?
Agent: Hello! I'm here and ready to assist you. How can I help you today?

 User: Hi. How are you doing today?
Agent: Hi there! I'm here to help you with any questions or issues you might have. What can I assist you with today?

 User: Hi. What is your name?
Agent: I'm your friendly customer support assistant. You can just call me Support. How can I assist you today?

 User: Hi! My system's name is HP
Agent: Hi! It's great to know that you have an HP system. How can I assist you with it today? Are you experiencing any issues or do you have questions?

 User: What is the name of my system?
Agent: You mentioned that your system's name is HP. If you need help with anything specific related to your HP system, feel free to ask!

 User: Hi. What is your name?
Agent: Hello! You can call me Support. I'm here to help you with any questions or issues you might have. How can I assist you today?

############################

## Step 9: Multi-Turn Conversation

In [35]:
# First message
run_conversation('Hi! My system\'s name is HP')


 User: Hi. How are you doing today?

 User: Hi. How are you doing today?
Agent: Hello! I'm here and ready to assist you. How can I help you today?

 User: Hi. How are you doing today?
Agent: Hi there! I'm here to help you with any questions or issues you might have. What can I assist you with today?

 User: Hi. What is your name?
Agent: I'm your friendly customer support assistant. You can just call me Support. How can I assist you today?

 User: Hi! My system's name is HP
Agent: Hi! It's great to know that you have an HP system. How can I assist you with it today? Are you experiencing any issues or do you have questions?

######################################################################


In [36]:
# Follow up question
run_conversation('What is the name of my system?')


 User: Hi. How are you doing today?

 User: Hi. How are you doing today?
Agent: Hello! I'm here and ready to assist you. How can I help you today?

 User: Hi. How are you doing today?
Agent: Hi there! I'm here to help you with any questions or issues you might have. What can I assist you with today?

 User: Hi. What is your name?
Agent: I'm your friendly customer support assistant. You can just call me Support. How can I assist you today?

 User: Hi! My system's name is HP
Agent: Hi! It's great to know that you have an HP system. How can I assist you with it today? Are you experiencing any issues or do you have questions?

 User: What is the name of my system?
Agent: You mentioned that your system's name is HP. If you need help with anything specific related to your HP system, feel free to ask!

######################################################################


In [37]:
run_conversation('What is he name of my system?', thread_id='0012')


 User: What is he name of my system?
Agent: I'm sorry, but I don't have access to specific information about your system or device. However, you can usually find the name of your system in the settings or about section of your device. If you let me know what type of system or device you're using, I can guide you on how to find that information!

######################################################################


In [43]:
def interactive_chat():
    """
    Run an interactive chat session.
    Type 'exit' or 'quit' to stop.
    """
    print('\n' + '*'*100)
    print('Interactive chat started')
    print("Type your message and press Enter. Type 'exit' to quit.")
    print('\n' + '*'*100)

    thread_id = 'interactive_session1'

    while True:
        user_input = input('You: ').strip()

        if user_input.lower() in ['exit', 'quit']:
            print('Thank you for using this service!\nBye for now')
            break

        if not user_input:
            continue

        # Get response
        result = agent.invoke(
            {'messages': [HumanMessage(content=user_input)]},
            config={'configurable':{'thread_id':thread_id}}
        )

        # Print agent's response
        agent_message = result['messages'][-1]
        print(f'\nAgent: {agent_message.content}')


interactive_chat()


****************************************************************************************************
Interactive chat started
Type your message and press Enter. Type 'exit' to quit.

****************************************************************************************************

Agent: Hi Dee! It’s great to meet you. How can I assist you today?

Agent: It's important to be cautious with any medication. Could you please describe the anomalies you noticed with the aspirin? That way, I can help you determine the best course of action. However, I always recommend consulting with a healthcare professional or pharmacist for the safest advice regarding medications.

Agent: Given the tampering and the fact that the expiry date is so close, I would strongly advise against using the aspirin. Taking medication that appears to be tampered with or is close to expiration can pose health risks. It would be best to return it to the pharmacist or the place of purchase for a replacement or refund.